Member 2 – Data Quality & Cleaning


Your main question is:

"What problems exist in the raw data, and how should we correctly handle them?"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

csv_file_path = '/content/drive/MyDrive/hotel-booking-cancellation/data/raw/hotel_bookings.csv'
df = pd.read_csv(csv_file_path)

display(df.head())

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


1. Check Missing & Null Values

In [ ]:
# Calculate total and percentage of missing values
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100


missing_summary = pd.DataFrame({
    'Missing Values': missing_data,
    'Percentage (%)': missing_percent
})

# Display only columns that have missing values, sorted highest to lowest
display(missing_summary[missing_summary['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False))

,Missing Values,Percentage (%)
company,112593,94.306893
agent,16340,13.686238
country,488,0.408744
children,4,0.003350


2. Identify Hidden "Undefined" Values

In [ ]:
# Count 'Undefined' strings in categorical columns
print("Undefined in meal:", (df['meal'] == 'Undefined').sum())
print("Undefined in market_segment:", (df['market_segment'] == 'Undefined').sum())
print("Undefined in distribution_channel:", (df['distribution_channel'] == 'Undefined').sum())

Undefined in meal: 1169
Undefined in market_segment: 2
Undefined in distribution_channel: 5


meal mapped to SC and other  2 coloumns mapped to unkown

3. Investigate Duplicates

In [ ]:

duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Total exact duplicate rows: {duplicate_count}")
print(f"Percentage of dataset: {duplicate_percentage:.2f}%")

Total exact duplicate rows: 31994
Percentage of dataset: 26.80%


after some cleaning transformation i removed 32001 duplicates

4. Check for Impossible or Suspicious Values

In [ ]:
# Check for bookings with 0 total guests
zero_guests = df[(df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)]
print(f"Bookings with 0 total guests: {len(zero_guests)}")

# Check for bookings with 0 total nights
zero_nights = df[(df['stays_in_weekend_nights'] == 0) & (df['stays_in_week_nights'] == 0)]
print(f"Bookings with 0 total nights: {len(zero_nights)}")

# Check for negative or zero Average Daily Rate (ADR)
print(f"Negative ADR values: {len(df[df['adr'] < 0])}")
print(f"Zero ADR values: {len(df[df['adr'] == 0])}")

Bookings with 0 total guests: 180
Bookings with 0 total nights: 715
Negative ADR values: 1
Zero ADR values: 1959


0 guests were removed,negetive ADR (Avarage daliy rate) was removed.

5. Investigate Outliers

In [ ]:
# Display summary statistics for key numerical columns to spot extreme max values
columns_to_check = ['lead_time', 'adr', 'adults', 'children', 'babies',
                    'stays_in_weekend_nights', 'stays_in_week_nights']

display(df[columns_to_check].describe())

Extreme lead times (>500 days): 481
Very high ADR (>500): 3
Bookings with more than 4 adults: 16
Bookings with more than 5 children: 1
Bookings with more than 5 babies: 2
Week stays longer than 30 nights: 9
Weekend stays longer than 10 nights: 15


,hotel,adr,adults,children,babies,customer_type,market_segment
48515,City Hotel,5400.00,2,0.0,0,Transient,Offline TA/TO
111403,City Hotel,510.00,1,0.0,0,Transient,Offline TA/TO
15083,Resort Hotel,508.00,2,0.0,0,Transient,Corporate
103912,City Hotel,451.50,2,2.0,0,Transient-Party,Direct
13142,Resort Hotel,450.00,2,0.0,0,Transient,Online TA
13391,Resort Hotel,437.00,2,2.0,0,Transient,Direct
39155,Resort Hotel,426.25,2,2.0,0,Transient,Direct
39568,Resort Hotel,402.00,3,1.0,0,Transient,Online TA
39118,Resort Hotel,397.38,3,2.0,0,Transient,Direct
13323,Resort Hotel,392.00,2,1.0,0,Transient,Direct


In [ ]:

# 5.1 INVESTIGATE EXTREME VALUES


print("Extreme lead times (>500 days):",
      (df['lead_time'] > 500).sum())

print("Very high ADR (>500):",
      (df['adr'] > 500).sum())

print("Bookings with more than 4 adults:",
      (df['adults'] > 4).sum())

print("Bookings with more than 5 children:",
      (df['children'] > 5).sum())

print("Bookings with more than 5 babies:",
      (df['babies'] > 5).sum())

print("Week stays longer than 30 nights:",
      (df['stays_in_week_nights'] > 30).sum())

print("Weekend stays longer than 10 nights:",
      (df['stays_in_weekend_nights'] > 10).sum())

# Show the highest ADR records for inspection
display(
    df.nlargest(10, 'adr')[
        ['hotel', 'adr', 'adults', 'children',
         'babies', 'customer_type', 'market_segment']
    ]
)

Extreme lead times (>500 days): 481
Very high ADR (>500): 3
Bookings with more than 4 adults: 16
Bookings with more than 5 children: 1
Bookings with more than 5 babies: 2
Week stays longer than 30 nights: 9
Weekend stays longer than 10 nights: 15


,hotel,adr,adults,children,babies,customer_type,market_segment
48515,City Hotel,5400.00,2,0.0,0,Transient,Offline TA/TO
111403,City Hotel,510.00,1,0.0,0,Transient,Offline TA/TO
15083,Resort Hotel,508.00,2,0.0,0,Transient,Corporate
103912,City Hotel,451.50,2,2.0,0,Transient-Party,Direct
13142,Resort Hotel,450.00,2,0.0,0,Transient,Online TA
13391,Resort Hotel,437.00,2,2.0,0,Transient,Direct
39155,Resort Hotel,426.25,2,2.0,0,Transient,Direct
39568,Resort Hotel,402.00,3,1.0,0,Transient,Online TA
39118,Resort Hotel,397.38,3,2.0,0,Transient,Direct
13323,Resort Hotel,392.00,2,1.0,0,Transient,Direct


Outlier decision: Extreme numerical values were investigated rather than automatically removed. Large values in lead time, guest counts and stay duration may represent legitimate long-term reservations or group bookings. Therefore, these observations were retained unless there was a clear logical reason to classify them as invalid. The single negative ADR value was removed because a negative average daily room rate was considered invalid. Zero ADR values were retained because they may represent complimentary or special bookings.


# 5.2 CHECK DATA TYPES


In [ ]:


print("DATA TYPES")
print("=" * 50)

df.info()

# Create a clearer summary
data_type_summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.astype(str),
    'Unique Values': df.nunique().values
})

display(data_type_summary)

DATA TYPES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal 

,Column,Data Type,Unique Values
hotel,hotel,object,2
is_canceled,is_canceled,int64,2
lead_time,lead_time,int64,479
arrival_date_year,arrival_date_year,int64,3
arrival_date_month,arrival_date_month,object,12
arrival_date_week_number,arrival_date_week_number,int64,53
arrival_date_day_of_month,arrival_date_day_of_month,int64,31
stays_in_weekend_nights,stays_in_weekend_nights,int64,17
stays_in_week_nights,stays_in_week_nights,int64,35
adults,adults,int64,14


6. Data Cleaning


In [ ]:

# Work on a copy of the original dataframe
df_clean = df.copy()

print("Original shape:", df_clean.shape)


# 6.1 Handle missing values


# children: only a very small number of missing values
df_clean['children'] = df_clean['children'].fillna(0)

# country: retain records and label missing values explicitly
df_clean['country'] = df_clean['country'].fillna('Unknown')

# agent: missing value interpreted as no/unknown travel agent
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['agent'] = df_clean['agent'].astype(int).astype(str)
df_clean['agent'] = df_clean['agent'].replace('0', 'No_Agent')

# company: remove because approximately 94% of values are missing
df_clean = df_clean.drop(columns=['company'])


# 6.2 Handle hidden "Undefined" values


# According to the dataset description, Undefined meal and SC
# both represent no meal package
df_clean['meal'] = df_clean['meal'].replace('Undefined', 'SC')

# Very small number of undefined values in these categorical columns
df_clean['market_segment'] = df_clean['market_segment'].replace(
    'Undefined', 'Unknown'
)

df_clean['distribution_channel'] = df_clean[
    'distribution_channel'
].replace('Undefined', 'Unknown')


# 6.3 Remove exact duplicate rows


rows_before_duplicates = len(df_clean)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

rows_after_duplicates = len(df_clean)

print(
    "Exact duplicates removed:",
    rows_before_duplicates - rows_after_duplicates
)


# 6.4 Remove bookings with zero guests


zero_guest_condition = (
    (df_clean['adults'] == 0) &
    (df_clean['children'] == 0) &
    (df_clean['babies'] == 0)
)

print(
    "Zero-guest bookings removed:",
    zero_guest_condition.sum()
)

df_clean = df_clean[~zero_guest_condition].copy()


# 6.5 Remove negative ADR


print(
    "Negative ADR records removed:",
    (df_clean['adr'] < 0).sum()
)

df_clean = df_clean[df_clean['adr'] >= 0].copy()

# ------------------------------------------------------------
# Note:
# Zero ADR values are retained because they may represent
# complimentary/special bookings.
#
# Zero-night bookings and unusual numerical outliers are
# investigated but not automatically removed without evidence
# that they are invalid.
# ------------------------------------------------------------

df_clean = df_clean.reset_index(drop=True)

print("\nCleaning completed.")
print("Final shape:", df_clean.shape)

Original shape: (119390, 32)
Exact duplicates removed: 32001
Zero-guest bookings removed: 166
Negative ADR records removed: 1

Cleaning completed.
Final shape: (87222, 31)


7. FINAL DATA QUALITY CHECK

In [ ]:


print("FINAL DATA QUALITY CHECK")
print("=" * 50)

print("\nDataset shape:")
print(df_clean.shape)

print("\nRemaining missing values:")
remaining_missing = df_clean.isnull().sum()
print(remaining_missing[remaining_missing > 0])

print("\nExact duplicate rows:")
print(df_clean.duplicated().sum())

print("\nZero-guest bookings:")
print(
    len(
        df_clean[
            (df_clean['adults'] == 0) &
            (df_clean['children'] == 0) &
            (df_clean['babies'] == 0)
        ]
    )
)

print("\nNegative ADR values:")
print((df_clean['adr'] < 0).sum())

print("\nZero ADR values retained:")
print((df_clean['adr'] == 0).sum())

print("\nUndefined categorical values remaining:")
print((df_clean == 'Undefined').sum().sum())

FINAL DATA QUALITY CHECK

Dataset shape:
(87222, 31)

Remaining missing values:
Series([], dtype: int64)

Exact duplicate rows:
0

Zero-guest bookings:
0

Negative ADR values:
0

Zero ADR values retained:
1643

Undefined categorical values remaining:
0


In [ ]:

# CLEANING DECISION SUMMARY


cleaning_summary = pd.DataFrame({
    'Data Quality Issue': [
        '4 missing children values',
        '488 missing country values',
        '16,340 missing agent values',
        '94.3% missing company values',
        'Undefined meal values',
        'Undefined market segment/channel values',
        'Exact duplicate rows',
        'Zero-guest bookings',
        'Zero-night bookings',
        'Negative ADR',
        'Zero ADR',
        'Extreme numerical values'
    ],

    'Decision': [
        'Replaced with 0',
        'Replaced with Unknown',
        'Represented as No_Agent',
        'Company column removed',
        'Mapped Undefined to SC',
        'Mapped Undefined to Unknown',
        'Removed after investigation',
        'Removed as invalid',
        'Retained for now',
        'Removed as invalid',
        'Retained',
        'Investigated and retained unless clearly invalid'
    ],

    'Reason': [
        'Only four missing values and zero is the dominant value',
        'Preserves booking records while representing unavailable country information',
        'Avoids numerical imputation of an identifier',
        'Extremely high missingness makes reliable use difficult',
        'Undefined and SC both represent no meal package',
        'Only a very small number of records are affected',
        'Prevent identical records appearing repeatedly and potentially affecting modelling',
        'A valid hotel reservation should contain at least one guest',
        'Could represent cancellations or unusual legitimate reservation records',
        'Negative room rate is not logically valid',
        'May represent complimentary or special bookings',
        'Unusual does not automatically mean erroneous'
    ]
})

display(cleaning_summary)

,Data Quality Issue,Decision,Reason
0,4 missing children values,Replaced with 0,Only four missing values and zero is the domin...
1,488 missing country values,Replaced with Unknown,Preserves booking records while representing u...
2,"16,340 missing agent values",Represented as No_Agent,Avoids numerical imputation of an identifier
3,94.3% missing company values,Company column removed,Extremely high missingness makes reliable use ...
4,Undefined meal values,Mapped Undefined to SC,Undefined and SC both represent no meal package
5,Undefined market segment/channel values,Mapped Undefined to Unknown,Only a very small number of records are affected
6,Exact duplicate rows,Removed after investigation,Prevent identical records appearing repeatedly...
7,Zero-guest bookings,Removed as invalid,A valid hotel reservation should contain at le...
8,Zero-night bookings,Retained for now,Could represent cancellations or unusual legit...
9,Negative ADR,Removed as invalid,Negative room rate is not logically valid


 8. SAVE CLEANED DATASET


In [ ]:


output_path = "/content/drive/MyDrive/hotel-booking-cancellation/data/processed/hotel_bookings_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("Location:", output_path)
print("Final shape:", df_clean.shape)

Cleaned dataset saved successfully.
Location: /content/drive/MyDrive/hotel-booking-cancellation/data/processed/hotel_bookings_cleaned.csv
Final shape: (87222, 31)


27.53 CANCELETION RATE AFTER CLEANING
BEFORE 37.04